# 04 – Tap, gradienter og læringsrate

**Tid:** 75 minutter. **Mål:** forklare tap og gradienter, oppdatere en parameter
og forstå de sentrale stegene i læring. Ingen derivasjonsregler forutsettes.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


## Eksempel: tap som avstand til et mål

Vi vil at parameteren `vekt` skal bli 3. Tapet er `(vekt - 3)²`: det er lite
når vekten er nær målet og stort langt unna. Den lokale helningen kalles
gradienten. PyTorch finner den når vi bruker `requires_grad=True` og `backward()`.
Positiv gradient betyr at et lite steg mot større vekt øker tapet.


In [1]:
verdier = torch.linspace(-1, 7, 100)
figur, akse = plt.subplots()
akse.plot(verdier, (verdier - 3) ** 2)
akse.set(xlabel="Parameterverdi", ylabel="Tap", title="Lavest tap ved parameterverdien 3")
plt.show()

vekt = torch.tensor(0.0, requires_grad=True)
tap = (vekt - 3) ** 2
tap.backward()
print("Vekt:", vekt.item(), "Tap:", tap.item(), "Gradient:", vekt.grad.item())

# Oppdateringen skal ikke bli en del av beregningen vi deriverer neste gang.
with torch.no_grad():
    vekt -= 0.1 * vekt.grad
vekt.grad.zero_()
print("Ny vekt:", vekt.item())


NameError: name 'torch' is not defined

## Oppgave 1: La én parameter lære

Start `parameter` på 0 med gradientsporing. Målet er nå **5**.
Kjør 20 oppdateringer med læringsrate 0.1 og tap `(parameter - 5) ** 2`.
Lagre tapene som vanlige Python-tall i `taphistorikk` og vis dem i en kurve.
Skriv ut den endelige parameterverdien. Nullstill gradienten etter hver oppdatering.

<details>
<summary>Hint</summary>

I løkken: beregn tap, kall `backward()`, oppdater under `torch.no_grad()`
og kall `parameter.grad.zero_()`. Lagre `tap.item()` i listen.

</details>


In [ ]:
# Skriv din løsning her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert len(taphistorikk) == 20
assert taphistorikk[-1] < taphistorikk[0]
assert abs(parameter.item() - 5) < 0.1


### Sjekk forståelsen

Var den første gradienten positiv eller negativ? Forklar hvorfor oppdateringen
flytter parameteren mot 5. Hva gjør henholdsvis `backward()` og selve oppdateringen?
Hvorfor nullstiller vi gradienten?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

...


## Oppgave 2: Sammenlign læringsrater

Gjenta forsøket fra parameterverdien 0 med læringsratene **0.01, 0.1 og 1.1**,
med 20 oppdateringer hver. Lag en ny parameter for hvert forsøk.
Vis alle tapskurvene i én figur med tegnforklaring. Bruk gjerne logaritmisk y-akse.
Skriv ut siste tap for hver læringsrate.

<details>
<summary>Hint</summary>

Bruk en ytre løkke over læringsratene og `akse.set_yscale("log")`.

</details>


In [ ]:
# Skriv din løsning her.


### Sjekk forståelsen

Beskriv hva du observerer for hver av de tre læringsratene.
Betyr en større læringsrate alltid raskere læring? Vis til resultatene dine.

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

...


## Eksempel: tap for klassifikasjon

MNIST trenger et tap for ti mulige klasser, ikke avstand mellom to siffertall.
`nn.CrossEntropyLoss` mottar **rå skårer** med form `[B, antall_klasser]`
og riktige klasseindekser som heltall med form `[B]`.
Vi bruker ikke softmax før denne tapsfunksjonen.

Eksemplet under har tre klasser og to eksempler. Høyere skår for riktig klasse
gir lavere tap når de andre skårene holdes uendret.


In [ ]:
tapsfunksjon = nn.CrossEntropyLoss()
riktige_klasser = torch.tensor([0, 2])
svake_skaarer = torch.tensor([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
bedre_skaarer = torch.tensor([[3.0, 0.0, 0.0], [0.0, 0.0, 3.0]])
print("Tap med like skårer:", tapsfunksjon(svake_skaarer, riktige_klasser).item())
print("Tap med høyere riktig skår:", tapsfunksjon(bedre_skaarer, riktige_klasser).item())

# Optimizer erstatter den manuelle oppdateringen når vi får mange parametere.
parameter = nn.Parameter(torch.tensor(0.0))
optimizer = torch.optim.SGD([parameter], lr=0.1)
optimizer.zero_grad()
tap = (parameter - 3) ** 2
tap.backward()
optimizer.step()
print("Parameter etter ett optimizer-steg:", parameter.item())


## Oppgave 3: Undersøk klassifikasjonstapet

Lag to eksempler med ti skårer hver. Riktige etiketter skal være `[2, 7]`.
Beregn først `tap_like` når alle skårene er 0. Lag så en ny skårtabell der
riktig klasse har skår 4 og resten er 0. Beregn `tap_bedre` og skriv ut begge.

<details>
<summary>Hint</summary>

`torch.zeros(2, 10)` lager tabellen. Sett `[0, 2]` og `[1, 7]` til 4.

</details>


In [ ]:
# Skriv din løsning her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert tap_bedre < tap_like
assert etiketter.dtype == torch.int64


### Sjekk forståelsen

Hvorfor er ikke forskjellen mellom predikert siffer og riktig siffer et godt tap her?
Hva gjør `zero_grad()`, `backward()` og `step()` i en treningsrunde?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

...
